# Compute resources

This notebook is for when a global analysis runs out of memory or takes too long. We explain why the amount of data a global request reads can't be reduced but the memory it needs can, check how big a global request is, and show how to run one on a laptop, an HPC system or a cloud machine.

> **Before you start.** Global analysis of the downscaled data is slow, and it's the part of this dataset that takes the most experience with Python and Xarray. A decade of one variable over the whole globe reads about 17.5 GB, and the two examples here took about 14 and 23 minutes when we ran them. If a region answers your question, use one. If the GCM's own resolution is enough, the bias-corrected data on the coarse grid (Section 8 of [`subsetting-and-exporting.ipynb`](subsetting-and-exporting.ipynb)) reads about 17 times less for a global decade.

For the short version, see Section 7 of [`subsetting-and-exporting.ipynb`](subsetting-and-exporting.ipynb). If you come across a term you don't know, check the [glossary](https://github.com/carbonplan/sai-downscaling-data-utils/blob/main/GLOSSARY.md).

**Contents:**
1. Reads versus memory
2. How much a global request reads
3. Running a global analysis
4. Measured results
5. Another example: a map of change

## Setup

Follow the [installation instructions](https://github.com/carbonplan/sai-downscaling-data-utils#installation) in the README, then launch JupyterLab with `pixi run jupyter lab`. The cell below loads helper functions from the [`scripts/`](../scripts/README.md) folder, so open this notebook from inside the cloned repository.

In [1]:
import sys
from pathlib import Path

import flox  # noqa - xarray groupby speedup
import numpy as np
import xarray as xr

# The helper functions live in the repository's scripts/ folder.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "data_access.py").exists()), None)
if repo is None:
    raise RuntimeError("Open this notebook from inside the sai-downscaling-data-utils repository.")
sys.path.insert(0, str(repo / "scripts"))

from notebook_helpers import check_dates, first_decade, peak_memory_gb  # noqa: E402
from data_access import check_members, describe_request, load_downscaling_store  # noqa: E402

## 1. Reads versus memory

Analyses that cover the whole globe are where notebooks most often crash, because they ask for so much data. Moving to a bigger machine usually doesn't help, because memory isn't what's holding you back.

The amount of data a request reads is fixed. Data is stored in chunks, and every chunk your selection touches has to be downloaded and decompressed in full. Two variables over the whole globe for a decade read about 35 GB no matter what machine you use. Since each chunk holds about a year of data, the read grows with the time span you ask for: a single global day reads 1.6 GB, and a decade reads about 17.5 GB per variable.

What you can change is how much of that data you hold in memory at once:

| Code | What it holds in memory |
|---|---|
| `ds["tas"].sel(time=decade).load()` | All 15.2 GB of the decade at once, before anything is reduced (30.3 GB for `tas` and `pr` together). This is the most common cause of a crashed notebook. |
| `ds["tas"].sel(time=decade).resample(time="YE").mean().compute()` | A few 3.8 MB chunks at a time while it works, and then just the 41.5 MB of annual means it returns. On a 4 GB machine the full two-variable run used at most 1.88 GB (Section 4). |

You may also come across `chunks="auto"`. It isn't an alternative to reducing your data first: it's a setting for how data is opened, and it makes each piece of work much bigger (about 102 MB instead of 3.8 MB) without reading any less. `load_downscaling_store(...)` already opens the data with the right chunk size, so you don't need to change anything.

In short, a global analysis is limited by how fast the data can be read, not by how much memory you have. It will run on a laptop; it just takes as long as the reads take.

## 2. How much a global request reads

The cell below checks how much data the first global request in this notebook reads: `tas` and `pr` for one decade. Nothing is computed, so it finishes right away.

In [2]:
# ===== CUSTOMIZE: Choose the data for the global run =====
gcm_name = "CESM2-WACCM6"  # "CESM2-WACCM6" or "UKESM1-1-LL"
method_name = "bcsd"  # "bcsd" or "qdmsd"
scenario_name = "ssp245"  # "historical", "ssp245", "g6_1p5k" or "g6_1p5k_end"
member_name = "003"  # needs both variables; check_members(...) below lists the members that have them
product_name = "downscaled"  # or "debiased_coarse"
global_vars = ["tas", "pr"]
# ==========================================================

# No modification needed below
# Check which members have both variables: tas and pr share every member, tas and tasmax only share 006-010.
check_members(scenario_name, global_vars, gcm=gcm_name, method=method_name, product=product_name)
print()

total_read = 0
for v in global_vars:
    ds_v = load_downscaling_store(
        scenario_name, v, gcm=gcm_name, method=method_name, member=member_name,
        product=product_name,
    )
    total_read += describe_request(ds_v[v].sel(time=first_decade(ds_v)), f"Global, 10 yr, {v}")

print(f"\ncombined read for both variables: {total_read / 1e9:.1f} GB")
print("This is the floor. Reducing changes memory, not bytes read.")

CESM2-WACCM6/bcsd/ssp245: members containing tas, pr: 001, 002, 003, 004, 005, 006, 007, 008, 009, 010



Global, 10 yr, tas:
  shape          {'time': 3653, 'lat': 721, 'lon': 1440}
  logical size     15.171 GB
  chunks touched     4620  (3.8 MB each, store chunks)
  data read        17.484 GB


Global, 10 yr, pr:
  shape          {'time': 3653, 'lat': 721, 'lon': 1440}
  logical size     15.171 GB
  chunks touched     4620  (3.8 MB each, store chunks)
  data read        17.484 GB

combined read for both variables: 35.0 GB
This is the floor. Reducing changes memory, not bytes read.


## 3. Running a global analysis

The cell below reduces the data before computing anything, so the only thing held in memory at the end is the annual, area-weighted result. It reads about 35 GB, so it's turned off. Set `run_global_demo = True` to run it.

It needs very little memory (under 2 GB, see Section 4), so any machine that can run this notebook can run it: a laptop, a workstation, an HPC system or a cloud machine. What changes from one machine to the next is how long it takes. The time goes into downloading and decompressing chunks, so more CPUs and a faster connection help, and more memory doesn't. The data is stored on AWS in the `us-west-2` region, so machines close to it read fastest.

**Running it without keeping JupyterLab open.** A long run doesn't need an open notebook. Set `run_global_demo = True`, save the notebook, and run this from the repository folder in a terminal. It runs every cell and saves a copy with the results next to the original:

```bash
pixi run jupyter nbconvert --to notebook --execute notebooks/compute-resources.ipynb \
    --output compute-resources-results.ipynb
```

**On an HPC system**, run that command inside a batch job rather than on a login node. We use NCAR's [Derecho](https://ncar-hpc-docs.readthedocs.io/en/latest/compute-systems/derecho/) as the example. Other systems work the same way, but their scheduler options differ.

Derecho already has [pixi as a module](https://ncar-hpc-docs.readthedocs.io/en/latest/environment-and-software/user-environment/package-managers/pixi/), so you don't need to install it. Set up the repository once, on a login node:

```bash
module unload conda  # pixi can't be loaded alongside conda
module load pixi
git clone https://github.com/carbonplan/sai-downscaling-data-utils.git /glade/work/$USER/sai-downscaling-data-utils
cd /glade/work/$USER/sai-downscaling-data-utils
pixi install --frozen
```

Then save this job script as `global-analysis.pbs` and submit it with `qsub global-analysis.pbs`:

```bash
#!/bin/bash
#PBS -N global-analysis
#PBS -A <project_code>
#PBS -j oe
#PBS -q develop
#PBS -l walltime=04:00:00
#PBS -l select=1:ncpus=4:mem=8GB

module load pixi
cd /glade/work/$USER/sai-downscaling-data-utils
pixi run jupyter nbconvert --to notebook --execute notebooks/compute-resources.ipynb \
    --output compute-resources-results.ipynb
```

A few things about that script:

- Replace `<project_code>` with your NCAR project code.
- It uses the `develop` queue, which shares nodes and [charges](https://ncar-hpc-docs.readthedocs.io/en/latest/pbs/charging/) only for the 4 CPUs you ask for. The `main` queue gives every job whole 128-core nodes and charges for all of them, which is a waste for this job. `develop` jobs can run for up to 6 hours, and you're charged for the time the job actually takes. On Casper, NCAR's data analysis cluster, use `-q casper` instead.
- The computation uses every CPU the job is given, so asking for more makes it faster. 8 GB of memory is plenty.
- The data is read over the internet. Before a long job, submit the script once with both global runs turned off. It only opens the data and checks sizes, so it finishes in about a minute, and if it works you know your jobs can reach the data.

To work interactively instead, go to [NCAR's JupyterHub](https://ncar-hpc-docs.readthedocs.io/en/latest/compute-systems/jupyterhub/), start a server on a Derecho batch node with the same queue and CPU settings, and pick the kernel named after this project (`sai-downscaling-data-utils`), which appears once `pixi install` has run. JupyterHub's file browser only shows your home directory, so link your work space there first: `ln -s /glade/work/$USER ~/work`.

The results we measured are in Section 4.

In [3]:
# ===== CUSTOMIZE: opt in to the full global run =====
run_global_demo = False  # True reads about 35 GB and takes 15 minutes or more
# =====================================================

# No modification needed below
if run_global_demo:
    import time

    t0 = time.perf_counter()
    reductions = {}
    for v in global_vars:
        ds_v = load_downscaling_store(
            scenario_name, v, gcm=gcm_name, method=method_name, member=member_name,
            product=product_name,
        )
        da_v = ds_v[v].sel(time=first_decade(ds_v))
        # Reduce FIRST. Annual means collapse ~3,653 daily steps into 10, and the
        # area-weighted spatial mean collapses the grid to a single number per year.
        annual_v = da_v.resample(time="YE").mean()
        weights_v = np.cos(np.deg2rad(da_v.lat))
        reductions[v] = annual_v.weighted(weights_v).mean(dim=["lat", "lon"])

    # Compute both variables together, so each is read only once.
    result = xr.Dataset(reductions).compute()
    peak_gb = peak_memory_gb()

    print(result)
    print(f"\npeak memory  {peak_gb:6.2f} GB" if peak_gb is not None else "\npeak memory  not available on Windows")
    print(f"time taken   {time.perf_counter() - t0:6.0f} s")
    print(f"result       {result.nbytes} bytes held, from {total_read / 1e9:.1f} GB read")
else:
    print("run_global_demo is False, so the ~35 GB global run was skipped.")
    print("The results we measured are in Section 4.")


run_global_demo is False, so the ~35 GB global run was skipped.
The results we measured are in Section 4.


## 4. Measured results

We ran the cell above on a cloud machine with 2 CPUs and 4 GB of memory, in the same AWS region as the data (`us-west-2`):

| Metric | Value |
|---|---|
| Variables | `tas`, `pr` (CESM2-WACCM6, bcsd, ssp245, member `003`) |
| Window | 2015-2024, global (`time=3653, lat=721, lon=1440`) |
| Logical size | 30.3 GB |
| Data read | 35.0 GB |
| **Peak memory** | **1.88 GB** |
| Time taken | 829 s (~14 min) |
| Size of the result | 160 bytes |

The run read 35 GB of data on a machine with only 4 GB of memory, and it never used more than 1.88 GB. That's because only a handful of 3.8 MB chunks are in memory at any one time, so the memory you need depends on the chunk size and how many chunks are processed at once, not on how much data you read. Calling `.load()` on the same data would need more than 30 GB of free memory.

The trade-off is time. The run took about 14 minutes, reading roughly 42 MB per second, and it was limited by decompressing data on 2 CPUs rather than by memory. More CPUs would make it faster; more memory would not. On a machine far from `us-west-2`, e.g. a laptop on a home connection or an HPC system in another part of the world, downloading can become the slowest step instead.

If you need to keep a map rather than a single number per year, reduce along time first. For example, `resample(time="YE").mean()` on a global decade gives 41.5 MB, which is easy to save to Zarr. Calling `.load()` on the full 30.3 GB is not. Section 5 shows a complete example.

## 5. Another example: a map of change

Many global analyses end in a map rather than a number per year. The cell below maps how much `tas` changes between 2020-2039 and 2050-2069, using the GCM, method, scenario and member from Section 2. It averages each 20-year period over time before computing anything, so the only thing held in memory at the end is one map of about 4 MB.

It reads about 67 GB, twice as much as the run in Section 3. A test run took about 23 minutes, which works out to about 48 MB per second. It's turned off too, and it prints how much it would read either way. The same map from the coarse bias-corrected data (`product_name = "debiased_coarse"` in Section 2) reads 2 to 4 GB.

In [4]:
# ===== CUSTOMIZE: Choose the variable and the two periods to compare =====
map_variable = "tas"
baseline_start = "2020-01-01"
baseline_end = "2039-12-31"
future_start = "2050-01-01"
future_end = "2069-12-31"
run_change_map = False  # True reads about 67 GB and takes 25 minutes or more
# ==========================================================================

# No modification needed below
ds_map = load_downscaling_store(
    scenario_name, map_variable, gcm=gcm_name, method=method_name, member=member_name,
    product=product_name,
)
check_dates(ds_map, baseline_start, baseline_end)
check_dates(ds_map, future_start, future_end)
baseline_days = ds_map[map_variable].sel(time=slice(baseline_start, baseline_end))
future_days = ds_map[map_variable].sel(time=slice(future_start, future_end))

map_read = describe_request(baseline_days, "baseline period") + describe_request(future_days, "future period")
print(f"\ncombined read for the map: {map_read / 1e9:.1f} GB")

if run_change_map:
    import time

    import matplotlib.pyplot as plt

    t0 = time.perf_counter()
    # Average each period over time first, so each becomes a single map before anything is computed.
    change = (future_days.mean("time") - baseline_days.mean("time")).compute()
    print(f"time taken   {time.perf_counter() - t0:6.0f} s")
    print(f"result       {change.nbytes / 1e6:.1f} MB held")

    limit = float(abs(change).max())
    units = ds_map[map_variable].attrs.get("units", "")
    change.plot(
        figsize=(11, 5), cmap="RdBu_r", vmin=-limit, vmax=limit,
        cbar_kwargs={"label": f"change in {map_variable} ({units})"},
    )
    plt.title(f"{gcm_name} {scenario_name} {method_name} {map_variable}: "
              f"{future_start} to {future_end} minus {baseline_start} to {baseline_end}")
    plt.show()
else:
    print("run_change_map is False, so the global map was skipped.")


baseline period:
  shape          {'time': 7305, 'lat': 721, 'lon': 1440}
  logical size     30.337 GB
  chunks touched     8820  (3.8 MB each, store chunks)
  data read        33.378 GB
future period:
  shape          {'time': 7305, 'lat': 721, 'lon': 1440}
  logical size     30.337 GB
  chunks touched     8820  (3.8 MB each, store chunks)
  data read        33.378 GB

combined read for the map: 66.8 GB
run_change_map is False, so the global map was skipped.
